# EarningsLens — Multi-Agent Earnings Call Credibility Auditor
## CT2 Group Assignment | AMPBA Batch 24
**Pipeline:** LangGraph | **LLM:** GPT-4o | **Dataset:** 18,755 transcripts

This notebook demonstrates a multi-agent system that audits 
management credibility on earnings calls by:
1. Loading current + prior quarter transcripts from local dataset
2. Extracting forward-looking guidance from prior quarter
3. Extracting reported actuals from current quarter  
4. Cross-validating guidance vs actuals with credibility scoring
5. Generating a structured analyst brief (Red Flag or Clean Bill)

## Section 1: Setup & Imports

In [ ]:
import sys, os

# Ensure working dir is earnings-lens/ regardless of where notebook is launched
if os.path.basename(os.getcwd()) != "earnings-lens":
    os.chdir("earnings-lens")

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from dotenv import load_dotenv
load_dotenv()

from pipeline.graph import run_pipeline
from pipeline.state import create_initial_state
import json, pandas as pd
from IPython.display import Markdown, display

# Confirm dataset loaded
from utils.pkl_loader import load_dataframe, get_available_tickers
df = load_dataframe("data/transcripts.pkl")
print(f"✅ Dataset loaded: {df.shape[0]:,} transcripts, "
      f"{len(get_available_tickers(df)):,} unique tickers")
print("✅ LLM: GPT-4o via OpenAI")
print("✅ Framework: LangGraph StateGraph")
print("✅ Pipeline: 5 agents + input/output guardrails")

✅ Dataset loaded: 18,755 transcripts, 2,876 unique tickers
✅ LLM: GPT-4o via OpenAI
✅ Framework: LangGraph StateGraph
✅ Pipeline: 5 agents + input/output guardrails


## Section 2: Architecture Overview

In [ ]:
# Pipeline architecture summary
architecture = """
INPUT: ticker + quarter
    ↓
[Guardrail] input_guard     — validates ticker, quarter, blocks injection
    ↓
[Agent 1]  transcript_loader — loads current + prior transcript from PKL
    ↓
[Agent 2]  guidance_extractor — extracts forward guidance (LLM, few-shot)
    ↓
[Agent 3]  actuals_extractor  — extracts reported results (LLM, few-shot)
    ↓
[Agent 4]  credibility_scorer — cross-validates, scores 0-100 (LLM)
    ↓
[Conditional routing]
    score < 50  → RED FLAG path
    score >= 50 → CLEAN BILL path
    ↓
[Agent 5]  report_generator  — generates structured analyst brief (LLM)
    ↓
[Guardrail] output_guard     — validates schema, PII check
    ↓
OUTPUT: markdown analyst brief
"""
print(architecture)


INPUT: ticker + quarter
    ↓
[Guardrail] input_guard     — validates ticker, quarter, blocks injection
    ↓
[Agent 1]  transcript_loader — loads current + prior transcript from PKL
    ↓
[Agent 2]  guidance_extractor — extracts forward guidance (LLM, few-shot)
    ↓
[Agent 3]  actuals_extractor  — extracts reported results (LLM, few-shot)
    ↓
[Agent 4]  credibility_scorer — cross-validates, scores 0-100 (LLM)
    ↓
[Conditional routing]
    score < 50  → RED FLAG path
    score >= 50 → CLEAN BILL path
    ↓
[Agent 5]  report_generator  — generates structured analyst brief (LLM)
    ↓
[Guardrail] output_guard     — validates schema, PII check
    ↓
OUTPUT: markdown analyst brief



## Section 3: Scenario 1 — Happy Path (BILI Q2 2020)
**Test type:** Normal expected input  
**Expected:** Full pipeline runs, report generated, no errors

In [ ]:
print("Running Scenario 1: Happy Path — BILI 2020-Q2")
print("=" * 55)

result_s1 = run_pipeline("BILI", "2020-Q2")

print(f"✅ input_valid:       {result_s1['input_valid']}")
print(f"✅ guidance_items:    {len(result_s1['guidance_items'])} extracted")
print(f"✅ actual_items:      {len(result_s1['actual_items'])} extracted")
print(f"✅ credibility_score: {result_s1['credibility_score']}/100")
print(f"✅ route:             {result_s1['route']}")
print(f"✅ output_valid:      {result_s1['output_valid']}")
print(f"✅ errors:            {result_s1['errors']}")
print()
display(Markdown(result_s1['report']))

Running Scenario 1: Happy Path — BILI 2020-Q2


Multiple transcripts found for ticker=BILI quarter=2020-Q2; using first row.
Multiple transcripts found for ticker=BILI quarter=2020-Q1; using first row.


✅ input_valid:       True
✅ guidance_items:    8 extracted
✅ actual_items:      14 extracted
✅ credibility_score: 25.0/100
✅ route:             red_flag
✅ output_valid:      True
✅ errors:            []



# ⚠️ RED FLAG BRIEF — BILI 2020-Q2
**Credibility Score: 25.0/100**

## Executive Summary
In Q2 2020, BILI provided guidance on several key metrics, but failed to report actuals for many of them, leading to a significant credibility gap. While revenue and operating margin met or exceeded expectations, the lack of transparency on other metrics such as MAU targets and premium memberships raises concerns about management's reliability and transparency.

## Guidance vs Actuals
| Metric                | Guided                        | Actual          | Verdict  | Delta                                                                 |
|-----------------------|-------------------------------|-----------------|----------|-----------------------------------------------------------------------|
| Revenue Guidance      | RMB2.50B-RMB2.55B             | RMB2.6 billion  | DELIVERED| RMB2.6 billion exceeded the high end of guidance range RMB2.55 billion|
| MAU Target            | 220 million                   | No comparable actual reported | MISSED   | No comparable actual reported                                         |
| Premium Memberships   | 10 million                    | No comparable actual reported | MISSED   | No comparable actual reported                                         |
| Operating Margin      | 23%                           | 23%             | DELIVERED| Actual operating margin of 23% met the guidance                       |
| R&D Spend             | RMB297 million                | RMB331 million  | MISSED   | RMB331 million exceeded guidance by RMB34 million                     |
| Headcount Increase    | Increased headcount in general and administrative personnel | No comparable actual reported | MISSED   | No comparable actual reported                                         |
| Game Offerings        | 30 high quality games         | No comparable actual reported | MISSED   | No comparable actual reported                                         |
| Game Titles           | 8 titles                      | No comparable actual reported | MISSED   | No comparable actual reported                                         |

## Key Misses
- **MAU Target**: The absence of reported actuals for the MAU target of 220 million suggests potential issues in user growth or data transparency.
- **Premium Memberships**: Failure to report on premium memberships raises questions about the success of monetization strategies.
- **R&D Spend**: Exceeding the R&D spend guidance by RMB34 million could indicate mismanagement of resources or unexpected project costs.
- **Headcount Increase**: Lack of reporting on headcount changes may reflect poor internal communication or strategic shifts.
- **Game Offerings and Titles**: Not reporting on game offerings and titles could imply delays or strategic pivots in the gaming segment.

## Language Drift Analysis
The use of hedging phrases such as "projecting," "target," "exceeding," "reaching," "was," "due to," "will be further expanding," and "scheduled to be released" suggests a lack of confidence and certainty in management's guidance. These terms indicate potential over-reliance on future expectations rather than current performance.

## Investment Implications
The significant credibility miss in BILI's Q2 2020 results suggests caution for investors. The lack of transparency and failure to report on key metrics could undermine investor confidence and lead to increased volatility in the stock. Investors should closely monitor future disclosures and management's ability to meet its guidance.

## Disclaimer
This brief is generated by EarningsLens for research purposes only. Not financial or investment advice.

## Section 4: Scenario 2 — Different Company (Cross-Quarter Validation)
**Test type:** Different company, tests Q1→Q4 prior year rollover  
**Expected:** Pipeline finds prior quarter, computes credibility score

In [ ]:
# Pick a strong ticker-quarter pair where prior quarter is also available
from utils.pkl_loader import get_available_tickers, get_available_quarters, get_prior_quarter

preferred_tickers = ["AAPL", "MSFT", "AMZN", "NVDA", "TSLA", "META", "GFF"]
all_tickers = get_available_tickers(df)

# Build candidate list: preferred first, then all others (excluding 1-char tickers)
candidate_tickers = [t for t in preferred_tickers if t in all_tickers]
candidate_tickers += [t for t in all_tickers if t not in candidate_tickers and len(t) >= 2]

result_s2 = None
s2_ticker, s2_quarter = None, None

for tk in candidate_tickers:
    quarters = sorted(get_available_quarters(df, tk), reverse=True)
    for q in quarters:
        try:
            pq = get_prior_quarter(q)
        except ValueError:
            continue
        if pq not in quarters:
            continue

        trial = run_pipeline(tk, q)
        # Choose a pair that is valid and produces at least some extracted signal
        if trial.get("input_valid") and (
            len(trial.get("guidance_items", [])) > 0 or len(trial.get("actual_items", [])) > 0
        ):
            s2_ticker, s2_quarter = tk, q
            result_s2 = trial
            break
    if result_s2 is not None:
        break

if result_s2 is None:
    raise ValueError("No robust cross-quarter pair found in dataset")

print(f"Running Scenario 2: Cross-Quarter — {s2_ticker} {s2_quarter}")
print("=" * 55)

print(f"input_valid:       {result_s2['input_valid']}")
print(f"prior_quarter:     {result_s2.get('prior_quarter', 'N/A')}")
print(f"guidance_items:    {len(result_s2.get('guidance_items', []))}")
print(f"actual_items:      {len(result_s2.get('actual_items', []))}")
print(f"credibility_score: {result_s2.get('credibility_score', 'N/A')}")
print(f"route:             {result_s2.get('route', 'N/A')}")
print(f"errors:            {result_s2['errors']}")
if result_s2.get('report'):
    display(Markdown(result_s2['report']))

Running Scenario 2: Cross-Quarter — AAPL 2023-Q1
input_valid:       True
prior_quarter:     2022-Q4
guidance_items:    9
actual_items:      14
credibility_score: 33.3
route:             red_flag
errors:            []


# ⚠️ RED FLAG BRIEF — AAPL 2023-Q1
**Credibility Score: 33.3/100**

## Executive Summary
In Q1 2023, Apple Inc. (AAPL) faced significant challenges in meeting its guidance, with several key metrics missed or lacking comparable actuals. The company refrained from providing revenue guidance, and while some areas like Mac revenue and gross margin were delivered as expected, the overall credibility of management's guidance is questionable due to multiple misses and lack of transparency in reporting.

## Guidance vs Actuals
| Metric | Guided | Actual | Verdict | Delta |
|--------|--------|--------|---------|-------|
| Revenue Guidance | Not providing revenue guidance | $117.2B, -5% YoY | MISSED | No comparable actual reported |
| Revenue Performance | Will decelerate | $117.2B, -5% YoY | DELIVERED | Revenue decreased by 5% YoY, indicating deceleration |
| Foreign Exchange Impact | Nearly 10 percentage points | No comparable actual reported | MISSED | No comparable actual reported |
| Mac Revenue Guidance | Decline substantially year over year | $7.7B, -29% YoY | DELIVERED | Mac revenue declined by 29% YoY, which is substantial |
| Gross Margin Guidance | Between 42.5% and 43.5% | 43%, +70 bps QoQ | DELIVERED | 43% within guided range of 42.5%-43.5% |
| Operating Expenses Guidance | Between $14.7 billion and $14.9 billion | No comparable actual reported | MISSED | No comparable actual reported |
| Other Income and Expense Guidance | Around negative $300 million | No comparable actual reported | MISSED | No comparable actual reported |
| Tax Rate Guidance | Around 16.5% | No comparable actual reported | MISSED | No comparable actual reported |
| Dividend Per Share | $0.23 | No comparable actual reported | MISSED | No comparable actual reported |

## Key Misses
- **Revenue Guidance**: Lack of revenue guidance creates uncertainty about future performance.
- **Foreign Exchange Impact**: Absence of actual data on foreign exchange impact raises concerns about financial transparency.
- **Operating Expenses**: No actuals reported, making it difficult to assess cost management.
- **Other Income and Expense**: Missing actuals hinder understanding of financial health.
- **Tax Rate**: Lack of actual tax rate data affects the assessment of net income.
- **Dividend Per Share**: Absence of actual dividend data impacts investor income expectations.

## Language Drift Analysis
- **"Not providing"**: Indicates a lack of confidence or visibility into future performance.
- **"Believe"**: Suggests uncertainty or a lack of firm commitment.
- **"Expect"**: Implies anticipation rather than certainty, reflecting cautious optimism.
- **"Around"**: Signals approximation and potential variability in outcomes.

## Investment Implications
The credibility miss in Apple's Q1 2023 results suggests caution for investors. The lack of transparency and multiple missed metrics could lead to increased volatility and uncertainty in Apple's stock performance. Investors should closely monitor future guidance and actuals to reassess Apple's financial health and management reliability.

## Disclaimer
This brief is generated by EarningsLens for research purposes only. Not financial or investment advice.

## Section 5: Scenario 3 — Edge Case (Q1 Prior Quarter Rollover)
**Test type:** Edge case — Q1 requires rolling back to prior year Q4  
**Expected:** prior_quarter correctly computed as YYYY-Q4 of prior year

In [ ]:
print("Running Scenario 3: Edge Case — Q1 rollover")
print("=" * 55)

# Find a ticker that has Q1 AND prior year Q4 in dataset
# BILI has Q1 2020 — check if 2019-Q4 exists
from utils.pkl_loader import get_available_quarters
bili_quarters = get_available_quarters(df, "BILI")
print(f"BILI available quarters: {bili_quarters}")

# Pick best ticker for Q1 rollover test
test_ticker = "BILI"
test_quarter = "2020-Q1"

result_s3 = run_pipeline(test_ticker, test_quarter)

print(f"input_valid:    {result_s3['input_valid']}")
print(f"prior_quarter:  {result_s3.get('prior_quarter', 'N/A')}")
print(f"errors:         {result_s3['errors']}")

if result_s3['input_valid']:
    print(f"credibility_score: {result_s3.get('credibility_score')}")
    print(f"route:             {result_s3.get('route')}")
    if result_s3.get('report'):
        display(Markdown(result_s3['report']))
else:
    print(f"⚠️ Prior quarter not in dataset — graceful failure confirmed")
    print(f"   This is expected if 2019-Q4 is not in the Motley Fool dataset")

Multiple transcripts found for ticker=BILI quarter=2020-Q1; using first row.


Running Scenario 3: Edge Case — Q1 rollover
BILI available quarters: ['2018-Q3', '2019-Q4', '2020-Q1', '2020-Q2', '2020-Q4', '2021-Q1', '2021-Q2', '2021-Q3']
input_valid:    True
prior_quarter:  2019-Q4
errors:         []
credibility_score: 40.0
route:             red_flag


# ⚠️ RED FLAG BRIEF — BILI 2020-Q1
**Credibility Score: 40.0/100**

## Executive Summary
In the first quarter of 2020, BILI's management provided guidance on several key metrics, but there were notable discrepancies between the guidance and actual results. While revenue and gross profit margin exceeded expectations, the company failed to report comparable actuals for MAU targets, R&D expenses, and headcount increases. This inconsistency raises concerns about management's credibility and their ability to deliver on strategic commitments.

## Guidance vs Actuals
| Metric | Guided | Actual | Verdict | Delta |
|--------|--------|--------|---------|-------|
| MAU target | RMB220 million by 2021 | No comparable actual reported | MISSED | No comparable actual reported |
| Q1 revenue guidance | RMB2.15 billion - RMB2.20 billion | RMB2.3 billion | DELIVERED | RMB2.3 billion exceeded guided range of RMB2.15 billion - RMB2.20 billion |
| gross profit margin | 19.8% | 23% | DELIVERED | 23% exceeded guided margin of 19.8% |
| R&D expenses | increased headcount in R&D personnel | No comparable actual reported | MISSED | No comparable actual reported |
| headcount increase | increased headcount in sales and marketing personnel | No comparable actual reported | MISSED | No comparable actual reported |

## Key Misses
- **MAU Target**: The lack of reported actuals for the MAU target raises questions about the company's user growth strategy and transparency.
- **R&D Expenses**: Failure to report on R&D headcount increases suggests potential delays or issues in innovation and product development.
- **Headcount Increase**: The absence of data on sales and marketing personnel growth may indicate challenges in scaling operations or executing marketing strategies.

## Language Drift Analysis
The use of hedging phrases such as "subject to," "assumptions," "could cause," "not guarantees," "are subject to," "could affect," and "does not undertake any obligation" suggests a lack of confidence from management at the time of guidance. These phrases indicate uncertainty and potential variability in achieving the stated goals.

## Investment Implications
The credibility miss in BILI's Q1 2020 results highlights the need for investors to exercise caution. The discrepancies between guidance and actuals, particularly in strategic areas like user growth and personnel expansion, suggest potential risks in the company's execution capabilities. Investors should closely monitor future reports for improved transparency and alignment with guided metrics.

## Disclaimer
This brief is generated by EarningsLens for research purposes only. Not financial or investment advice.

## Section 6: Scenario 4 — Adversarial Input (Prompt Injection)
**Test type:** Adversarial — prompt injection attempt  
**Expected:** input_guard catches injection, pipeline blocked immediately

In [ ]:
print("Running Scenario 4: Adversarial — Prompt Injection")
print("=" * 55)

adversarial_inputs = [
    ("IGNORE PREVIOUS INSTRUCTIONS", "2020-Q2"),
    ("AAPL; DROP TABLE transcripts--", "2020-Q3"),
    ("act as", "2020-Q1"),
]

for ticker, quarter in adversarial_inputs:
    result = run_pipeline(ticker, quarter)
    status = "✅ BLOCKED" if not result['input_valid'] else "❌ NOT BLOCKED"
    print(f"{status} | ticker='{ticker}' | errors={result['errors']}")

Input guard failed with 2 error(s).
Input guard failed with 2 error(s).
Input guard failed with 2 error(s).


Running Scenario 4: Adversarial — Prompt Injection
✅ BLOCKED | ticker='IGNORE PREVIOUS INSTRUCTIONS' | errors=['Invalid ticker: must match ^[A-Za-z][A-Za-z0-9.\\-]{0,9}$ (examples: AAPL, BRK.B, BILI).', "Potential prompt injection detected: 'ignore previous'."]
✅ BLOCKED | ticker='AAPL; DROP TABLE transcripts--' | errors=['Invalid ticker: must match ^[A-Za-z][A-Za-z0-9.\\-]{0,9}$ (examples: AAPL, BRK.B, BILI).', "Potential prompt injection detected: 'drop table'."]
✅ BLOCKED | ticker='act as' | errors=['Invalid ticker: must match ^[A-Za-z][A-Za-z0-9.\\-]{0,9}$ (examples: AAPL, BRK.B, BILI).', "Potential prompt injection detected: 'act as'."]


## Section 7: Scenario 5 — Failure Recovery (Unknown Ticker)
**Test type:** Failure/recovery — ticker not in dataset  
**Expected:** Graceful error, no crash, meaningful error message

In [ ]:
print("Running Scenario 5: Failure Recovery — Unknown Ticker")
print("=" * 55)

failure_inputs = [
    ("ZZZZZ", "2020-Q2"),   # completely unknown ticker
    ("BILI",  "2015-Q1"),   # valid ticker, out-of-range quarter
    ("AAPL",  "9999-Q1"),   # year out of range — caught by guardrail
]

for ticker, quarter in failure_inputs:
    result = run_pipeline(ticker, quarter)
    if not result['input_valid'] or result['errors']:
        print(f"✅ Graceful failure | {ticker} {quarter}")
        print(f"   errors: {result['errors']}")
    else:
        print(f"ℹ️  Unexpected success | {ticker} {quarter}")
        print(f"   score: {result.get('credibility_score')}")

Output guard failed with 2 new error(s).
Output guard failed with 2 new error(s).
Input guard failed with 1 error(s).


Running Scenario 5: Failure Recovery — Unknown Ticker
✅ Graceful failure | ZZZZZ 2020-Q2
   errors: ['No transcript found for ZZZZZ 2020-Q2', 'Report must be a non-empty string with length > 50.', "Route must be one of: 'red_flag', 'clean_bill'."]
✅ Graceful failure | BILI 2015-Q1
   errors: ['No transcript found for BILI 2015-Q1', 'Report must be a non-empty string with length > 50.', "Route must be one of: 'red_flag', 'clean_bill'."]
✅ Graceful failure | AAPL 9999-Q1
   errors: ['Invalid quarter year: must be between 2000 and 2030.']


## Section 8: Scenario Results Summary

In [ ]:
scenarios = [
    {
        "scenario": "1 — Happy Path",
        "ticker": "BILI", "quarter": "2020-Q2",
        "type": "Happy path",
        "input_valid": result_s1['input_valid'],
        "score": result_s1.get('credibility_score', 'N/A'),
        "route": result_s1.get('route', 'N/A'),
        "status": "✅ PASS"
    },
    {
        "scenario": "2 — Cross-Quarter",
        "ticker": s2_ticker, "quarter": s2_quarter,
        "type": "Different company",
        "input_valid": result_s2['input_valid'],
        "score": result_s2.get('credibility_score', 'N/A'),
        "route": result_s2.get('route', 'N/A'),
        "status": "✅ PASS" if result_s2['input_valid'] else "⚠️ No prior quarter"
    },
    {
        "scenario": "3 — Q1 Rollover",
        "ticker": test_ticker, "quarter": test_quarter,
        "type": "Edge case",
        "input_valid": result_s3['input_valid'],
        "score": result_s3.get('credibility_score', 'N/A'),
        "route": result_s3.get('route', 'N/A'),
        "status": "✅ PASS"
    },
    {
        "scenario": "4 — Prompt Injection",
        "ticker": "IGNORE...", "quarter": "2020-Q2",
        "type": "Adversarial",
        "input_valid": False,
        "score": "N/A",
        "route": "N/A",
        "status": "✅ BLOCKED"
    },
    {
        "scenario": "5 — Unknown Ticker",
        "ticker": "ZZZZZ", "quarter": "2020-Q2",
        "type": "Failure/recovery",
        "input_valid": False,
        "score": "N/A",
        "route": "N/A",
        "status": "✅ GRACEFUL FAIL"
    },
]

summary_df = pd.DataFrame(scenarios)
display(summary_df[["scenario","type","input_valid","score","route","status"]])

,scenario,type,input_valid,score,route,status
0,1 — Happy Path,Happy path,True,25.0,red_flag,✅ PASS
1,2 — Cross-Quarter,Different company,True,33.3,red_flag,✅ PASS
2,3 — Q1 Rollover,Edge case,True,40.0,red_flag,✅ PASS
3,4 — Prompt Injection,Adversarial,False,N/A,N/A,✅ BLOCKED
4,5 — Unknown Ticker,Failure/recovery,False,N/A,N/A,✅ GRACEFUL FAIL


## Section 9: Sub-Agent Evaluation — Guidance Extractor

In [ ]:
# Run standalone evaluation of Agent 2 (guidance_extractor)
exec(open('evaluation/eval_runner.py').read())


GUIDANCE EXTRACTOR EVALUATION RESULTS
 ID Expected   Got  Precision   Recall     F1  Fail?
----------------------------------------------------------------------
  1        1     1       1.00     1.00   1.00      ✅
  2        1     1       1.00     1.00   1.00      ✅
  3        1     1       1.00     1.00   1.00      ✅
  4        1     1       1.00     1.00   1.00      ✅
  5        1     1       1.00     1.00   1.00      ✅
  6        3     3       1.00     1.00   1.00      ✅
  7        3     3       0.67     0.67   0.67      ✅
  8        3     3       1.00     1.00   1.00      ✅
  9        3     3       1.00     1.00   1.00      ✅
 10        3     3       0.67     0.67   0.67      ✅
 11        0     0       0.00     0.00   0.00      ✅
 12        0     0       0.00     0.00   0.00      ✅
 13        0     0       0.00     0.00   0.00      ✅
 14        0     0       0.00     0.00   0.00      ✅
 15        0     0       0.00     0.00   0.00      ✅
 16        0     0       0.00     0.00   0